# RTL Multi-Agent Evaluation on Colab/Kaggle

This notebook evaluates causal LLMs on RTL-generation benchmarks without fine-tuning.

It supports:

- VerilogEval v2
- RTLLM v2
- ChipBench-SC
- CVDP cid03

The notebook loads models one at a time, generates Verilog, runs syntax checks with `iverilog` when available, and runs functional simulation when the benchmark file includes a testbench.

Important: the true top coding models are usually too large for small hosted GPUs. This notebook therefore has two model lists:

- `TOP_HEAVY_MODELS`: the models you may want to test on A100/H100/API/vLLM.
- `ADVANCED_MODELS_TO_TEST`: advanced local/4-bit candidates; use `KAGGLE_FALLBACK_MODELS` if the runtime runs out of memory.


## Benchmark Data

This notebook first tries to download open benchmark/example files from GitHub, then searches local runtime inputs as fallback.

Default auto-download sources include ChipMATE, VerilogEval, RTLLM/RTL-Coder, ChipBench, and the Hugging Face CVDP dataset.

The ChipMATE repo provides `examples/verilogeval_v2.jsonl`. RTLLM files are commonly found in RTL-Coder-style repos and may use `.json` as JSONL. CVDP cid03 is loaded from `nvidia/cvdp-benchmark-dataset` via Hugging Face `datasets`, then filtered by `cid03`/`cid003` category tags. The notebook now requires all four benchmarks by default and stops with a clear missing-file list instead of silently running a partial evaluation.

Recognized benchmark filenames:

- `*verilogeval*v2*.jsonl/csv`
- `*rtllm*v2*.jsonl/csv`
- `*chipbench*sc*.jsonl/csv`
- `*cvdp*cid03*.jsonl/csv`

Recognized row fields:

- task id: `task_id`, `id`, `name`
- prompt/spec: `question`, `prompt`, `instruction`, `description`, `spec`, `problem`
- module/interface: `ref_sv`, `module_header`, `header`, `module`, `interface`
- testbench: `testbench`, `tb`, `test`, `tests`

If a row only has `question` and `ref_sv`, the notebook can still generate Verilog and syntax-check it. True functional pass@k requires a testbench or a benchmark-specific evaluator. Set `REQUIRE_ALL_BENCHMARKS=False` only if you intentionally want a partial smoke run.


In [ ]:
# Install runtime dependencies.
import os
import subprocess
import sys
from pathlib import Path

def run(cmd, check=False):
    print('$', cmd)
    return subprocess.run(cmd, shell=True, check=check)

run(f'{sys.executable} -m pip install --no-cache-dir -q -U transformers accelerate bitsandbytes datasets pandas tqdm huggingface_hub')

# Functional Verilog checks need iverilog/vvp. Colab and Kaggle usually allow apt-get.
# If this fails, the notebook will still run generation and mark syntax/sim as unavailable.
run('apt-get update -qq && apt-get install -y -qq iverilog git', check=False)


In [ ]:
import gc
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

try:
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

IS_KAGGLE = Path('/kaggle/input').exists()
IS_COLAB = ('google.colab' in sys.modules) or Path('/content').exists()

# For long Colab runs, set USE_GOOGLE_DRIVE_OUTPUT=True after mounting Drive if
# you want outputs/checkpoints to survive runtime disconnects.
USE_GOOGLE_DRIVE_OUTPUT = False
DRIVE_WORK_ROOT = Path('/content/drive/MyDrive/rtl_multi_agent_eval')

if IS_KAGGLE:
    INPUT_ROOT = Path('/kaggle/input')
    WORK_ROOT = Path('/kaggle/working')
    TEMP_ROOT = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else WORK_ROOT / '_temp'
elif IS_COLAB:
    INPUT_ROOT = Path('/content')
    WORK_ROOT = DRIVE_WORK_ROOT if USE_GOOGLE_DRIVE_OUTPUT else Path('/content')
    TEMP_ROOT = Path('/content/_temp')
else:
    INPUT_ROOT = Path.cwd()
    WORK_ROOT = Path.cwd()
    TEMP_ROOT = WORK_ROOT / '_temp'

OUT_DIR = WORK_ROOT / 'rtl_eval_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
BENCH_REPO_DIR = WORK_ROOT / 'rtl_benchmark_repos'
BENCH_REPO_DIR.mkdir(parents=True, exist_ok=True)

# Keep large model/dataset caches outside the saved output directory.
HF_CACHE_DIR = TEMP_ROOT / 'hf_cache'
CLEANUP_HF_CACHE_BETWEEN_MODELS = IS_KAGGLE
HF_DATASETS_CACHE = TEMP_ROOT / 'hf_datasets'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_DATASETS_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR / 'transformers')
os.environ['HF_DATASETS_CACHE'] = str(HF_DATASETS_CACHE)
os.environ['XDG_CACHE_HOME'] = str(TEMP_ROOT / 'xdg_cache')

print('Runtime:', 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local'))
print('Input root:', INPUT_ROOT)
print('Output dir:', OUT_DIR)
print('Benchmark repo dir:', BENCH_REPO_DIR)
print('HF cache dir:', HF_CACHE_DIR)
print('Clean HF cache between models:', CLEANUP_HF_CACHE_BETWEEN_MODELS)
print('HF datasets cache:', HF_DATASETS_CACHE)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name}, VRAM={props.total_memory/1e9:.1f} GB')
print('iverilog:', shutil.which('iverilog'))
print('vvp:', shutil.which('vvp'))


In [ ]:
# -----------------------------
# Model configuration
# -----------------------------

# True heavy/top coding-model candidates. Most of these will NOT fit on small hosted GPUs.
# Use these with A100/H100/vLLM or replace by an API endpoint.
TOP_HEAVY_MODELS = [
    'Qwen/Qwen3-Coder-480B-A35B-Instruct',
    'deepseek-ai/DeepSeek-V3.1',
    'moonshotai/Kimi-K2-Instruct',
    'deepseek-ai/DeepSeek-Coder-V2-Instruct',
    'Qwen/Qwen2.5-Coder-32B-Instruct',
]

# Stronger local/open-weight candidates to try in 4-bit. This list is
# intentionally more ambitious than the earlier top-3. Some entries may still
# fail on T4 x2 due to VRAM or gated/renamed model IDs.
ADVANCED_MODELS_TO_TEST = [
    # ChipMATE paper base model. Keep this for paper-faithful comparison.
    'Qwen/Qwen3.5-9B',
    # Stronger modern code model; may be too large for T4 x2 even in 4-bit.
    'Qwen/Qwen2.5-Coder-32B-Instruct',
    # More practical Qwen coder comparator.
    'Qwen/Qwen2.5-Coder-14B-Instruct',
    # DeepSeek coder family comparator.
    'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct',
    # Strong smaller fallback if the above fail.
    'Qwen/Qwen2.5-Coder-7B-Instruct',
]

# Practical fallback if the advanced list is too slow or too large.
KAGGLE_FALLBACK_MODELS = [
    'Qwen/Qwen3.5-9B',
    'Qwen/Qwen2.5-Coder-7B-Instruct',
    'deepseek-ai/deepseek-coder-6.7b-instruct',
    'bigcode/starcoder2-7b',
]

# ChipMATE paper base checkpoints. Removed Qwen3.5-4B from the default because
# you asked for stronger models, but it is kept here as an optional ablation.
CHIPMATE_BASE_MODELS = [
    'Qwen/Qwen3.5-9B',
    # 'Qwen/Qwen3.5-4B',
]

# Already trained ChipMATE Verilog agents from the paper. These are not base models,
# but they are useful paper baselines if you simply want to load and evaluate.
CHIPMATE_RELEASED_VERILOG_AGENTS = [
    'core12345/ChipMATE-V-9B',
    'core12345/ChipMATE-V-4B',
]

# Optimized Kaggle sweep: three practical coder models, smallest first.
# This avoids Qwen3.5-9B and 32B-class models while still comparing strong
# modern code backbones. If disk still fills, run one model per notebook session.
MODELS_TO_RUN = [
    'Qwen/Qwen2.5-Coder-7B-Instruct',
    'deepseek-ai/deepseek-coder-6.7b-instruct',
    'Qwen/Qwen2.5-Coder-14B-Instruct',
]
# One-model fallbacks:
# MODELS_TO_RUN = ['Qwen/Qwen2.5-Coder-7B-Instruct']
# MODELS_TO_RUN = ['deepseek-ai/deepseek-coder-6.7b-instruct']
# MODELS_TO_RUN = ['Qwen/Qwen2.5-Coder-14B-Instruct']

# Runtime controls. For the multi-agent VerilogEval ablation we load the
# full VerilogEval file first, then select a harder mixed subset below.
# If you enable RUN_BASELINE_SWEEP for all benchmarks, set this back to 50.
MAX_TASKS_PER_BENCHMARK = None
NUM_SAMPLES_PER_TASK = 1
MAX_NEW_TOKENS = 384
TOKENIZER_MAX_LENGTH = 4096
TEMPERATURE = 0.0
TOP_P = 0.95
LOAD_IN_4BIT = True
TRUST_REMOTE_CODE = True
SAVE_GENERATIONS = False
RUN_SIMULATION = False  # Syntax-only by default for speed; set True for testbench simulation.
RUN_BASELINE_SWEEP = False  # Baseline model sweep is already done; focus this run on the multi-agent ablation.

print('Models selected:')
for m in MODELS_TO_RUN:
    print(' -', m)
print('Task cap per benchmark:', MAX_TASKS_PER_BENCHMARK)
print('Max new tokens:', MAX_NEW_TOKENS, '| tokenizer max length:', TOKENIZER_MAX_LENGTH)
print('Run simulation:', RUN_SIMULATION)
print('Run baseline sweep:', RUN_BASELINE_SWEEP)


# -----------------------------
# VerilogEval multi-agent ablation
# -----------------------------
# This keeps the experiment runnable on Colab/Kaggle by default. The single-agent and
# multi-agent paths use the SAME RTL generator, so any pass@1 delta is from the
# verification/repair pipeline rather than a stronger RTL model.
RUN_MULTI_AGENT_VERILOGEVAL = True
MULTI_AGENT_MAX_TASKS = 75  # Larger, harder mixed subset. Use None for full VerilogEval.
MULTI_AGENT_TASK_SELECTION = 'complexity_weighted'  # early/mid/late mix, biased toward harder later tasks.

# Colab default for a cleaner comparison: the direct single-agent baseline
# and the repair stage both use the same 7B RTL-capable model. This tests
# whether the multi-agent oracle/repair loop helps beyond simply using 7B.
MULTI_RTL_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MULTI_ORACLE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-instruct'
MULTI_REPAIR_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MULTI_CRITIC_MODEL = 'bigcode/starcoder2-7b'

# Controlled ablation: same RTL generator, same task subset, same repair model,
# but different oracle/reference model family. This directly tests whether a
# same-family verifier is weaker because it shares the RTL generator's biases.
RUN_ORACLE_FAMILY_ABLATION = True
ORACLE_FAMILY_ABLATIONS = [
    {
        'name': 'same_family_qwen_oracle',
        'oracle_model': 'Qwen/Qwen2.5-Coder-7B-Instruct',
        'family_relation': 'same_family',
        'note': 'Qwen RTL generator + Qwen oracle/reference generator',
    },
    {
        'name': 'different_family_deepseek_oracle',
        'oracle_model': 'deepseek-ai/deepseek-coder-6.7b-instruct',
        'family_relation': 'different_family',
        'note': 'Qwen RTL generator + DeepSeek oracle/reference generator',
    },
]

# Stronger drop-in options when runtime/VRAM allow:
# Alternative stronger options when runtime/VRAM allow:
# MULTI_RTL_MODEL = 'Qwen/Qwen2.5-Coder-14B-Instruct'
# MULTI_ORACLE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'  # alternative oracle family
# MULTI_REPAIR_MODEL = 'Qwen/Qwen2.5-Coder-14B-Instruct'
# MULTI_CRITIC_MODEL = 'Qwen/QwQ-32B'  # Stronger reasoning critic; expensive.

MULTI_AGENT_USE_CRITIC = False  # Disabled for the oracle-family ablation to avoid a third-model confound.
MULTI_AGENT_MAX_REPAIR_ROUNDS = 1
MULTI_AGENT_REPAIR_TRIGGER = 'compile_or_reference_mismatch'  # Never use golden benchmark result to trigger repair.
MULTI_AGENT_MAX_NEW_TOKENS_RTL = 512
MULTI_AGENT_MAX_NEW_TOKENS_ORACLE = 1536
MULTI_AGENT_MAX_NEW_TOKENS_CRITIC = 768
MULTI_AGENT_RUN_GOLDEN_TB = True       # Uses benchmark testbench when present.
MULTI_AGENT_RUN_ORACLE_TB = True       # Uses generated self-checking testbench.
MULTI_AGENT_MUTATION_TEST = True       # Validates whether oracle tests catch simple RTL mutants.
MULTI_AGENT_MUTANTS_PER_TASK = 4

print('Multi-agent VerilogEval enabled:', RUN_MULTI_AGENT_VERILOGEVAL)
print('Multi-agent task cap:', MULTI_AGENT_MAX_TASKS)
print('Multi-agent RTL model:', MULTI_RTL_MODEL)
print('Multi-agent oracle model:', MULTI_ORACLE_MODEL)
print('Oracle family ablation enabled:', RUN_ORACLE_FAMILY_ABLATION)
if RUN_ORACLE_FAMILY_ABLATION:
    for variant in ORACLE_FAMILY_ABLATIONS:
        print(' - oracle variant:', variant['name'], '|', variant['family_relation'], '|', variant['oracle_model'])
print('Multi-agent repair model:', MULTI_REPAIR_MODEL)
print('Multi-agent task selection:', MULTI_AGENT_TASK_SELECTION)
print('Multi-agent critic model:', MULTI_CRITIC_MODEL if MULTI_AGENT_USE_CRITIC else '(disabled)')
print('Multi-agent repair trigger:', MULTI_AGENT_REPAIR_TRIGGER)


In [ ]:
# -----------------------------
# Auto-download benchmark repositories
# -----------------------------

AUTO_DOWNLOAD_BENCHMARKS = True
ACTIVE_BENCHMARK_NAMES = ['VerilogEval_v2'] if (RUN_MULTI_AGENT_VERILOGEVAL and not RUN_BASELINE_SWEEP) else None
REQUIRE_ALL_BENCHMARKS = False if ACTIVE_BENCHMARK_NAMES else True
BENCHMARK_REPOS = {
    # ChipMATE examples include VerilogEval V2 and may include additional paper eval files.
    'ChipMATE': 'https://github.com/zhongkaiyu/ChipMATE.git',
    # VerilogEval official benchmark repo.
    'verilog-eval': 'https://github.com/NVlabs/verilog-eval.git',
    # RTLLM / OpenLLM-RTL related public repos mentioned by RTL-Coder/OpenLLM-RTL.
    'RTLLM': 'https://github.com/hkust-zhiyao/RTLLM.git',
    'RTL-Coder': 'https://github.com/hkust-zhiyao/RTL-Coder.git',
    # Best-effort public repo guess for ChipBench. If it fails or does not contain
    # the data files, upload the missing benchmark JSONL/CSV as Kaggle input.
    'ChipBench': 'https://github.com/zhongkaiyu/ChipBench.git',
}

def clone_repo(name, url):
    dest = BENCH_REPO_DIR / name
    if dest.exists() and any(dest.iterdir()):
        print(f'{name} already exists:', dest)
        return dest
    print('Cloning', url, '->', dest)
    proc = subprocess.run(
        ['git', 'clone', '--depth', '1', url, str(dest)],
        capture_output=True,
        text=True,
        timeout=180,
    )
    if proc.returncode != 0:
        print(f'WARNING: clone failed for {name}:')
        print(proc.stderr[-2000:])
        return None
    return dest

if ACTIVE_BENCHMARK_NAMES == ['VerilogEval_v2']:
    BENCHMARK_REPOS = {k: v for k, v in BENCHMARK_REPOS.items() if k in {'ChipMATE', 'verilog-eval'}}

if AUTO_DOWNLOAD_BENCHMARKS:
    for name, url in tqdm(BENCHMARK_REPOS.items(), desc='Downloading benchmark repos'):
        clone_repo(name, url)

print('Active benchmark names:', ACTIVE_BENCHMARK_NAMES or 'all')
print('Require all benchmarks:', REQUIRE_ALL_BENCHMARKS)
print('Downloaded benchmark files:')
for p in sorted(BENCH_REPO_DIR.rglob('*')):
    if p.is_file() and p.suffix.lower() in {'.jsonl', '.json', '.csv'}:
        print(' -', p)


In [ ]:
# -----------------------------
# Benchmark loading
# -----------------------------

BENCHMARK_PATTERNS = {
    'VerilogEval_v2': ['*verilogeval*v2*.jsonl', '*verilogeval*.jsonl', '*verilogeval*v2*.csv', '*verilogeval*.csv'],
    'RTLLM_v2': ['*rtllm*v2*.jsonl', '*rtllm*.jsonl', '*rtllm*1.1*.json', '*rtllm*.json', '*rtllm*v2*.csv', '*rtllm*.csv'],
    'ChipBench_SC': ['*chipbench*sc*.jsonl', '*chipbench*.jsonl', '*chipbench*sc*.json', '*chipbench*.json', '*chipbench*sc*.csv', '*chipbench*.csv', '*data.jsonl'],
    'CVDP_cid03': ['*cvdp*cid03*.jsonl', '*cvdp*.jsonl', '*cvdp*cid03*.json', '*cvdp*.json', '*cvdp*cid03*.csv', '*cvdp*.csv'],
}

CVDP_HF_DATASET = 'nvidia/cvdp-benchmark-dataset'
CVDP_CATEGORY_TERMS = {'cid03', 'cid003'}
CVDP_HF_CONFIGS = None  # None means discover all configs automatically.


SEARCH_ROOTS = [BENCH_REPO_DIR, INPUT_ROOT, WORK_ROOT]
ID_KEYS = ['task_id', 'id', 'name', 'uid', 'instance_id', 'problem_id']
PROMPT_KEYS = ['question', 'prompt', 'instruction', 'Instruction', 'description', 'spec', 'problem']
CONTEXT_KEYS = ['context', 'repo_context', 'file_context']
HEADER_KEYS = ['ref_sv', 'module_header', 'header', 'module', 'interface', 'Input']
TESTBENCH_KEYS = ['testbench', 'tb', 'test', 'tests', 'test_bench', 'harness']

def find_candidates(patterns):
    seen = set()
    candidates = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for pattern in patterns:
            for p in root.rglob(pattern):
                if p.is_file() and p not in seen:
                    seen.add(p)
                    candidates.append(p)
    return candidates


def load_verilogeval_v2_from_official_files():
    """Load official NVlabs/verilog-eval v2 spec-to-RTL prompts and tests.

    ChipMATE's example JSONL is convenient for prompting, but it does not carry
    golden testbenches. The official repo stores each problem as:
      ProbXXX_name_prompt.txt
      ProbXXX_name_test.sv
      ProbXXX_name_ref.sv
    Loading this directory gives a real golden pass@1 signal.
    """
    rows = []
    seen = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for ds_dir in root.rglob('dataset_spec-to-rtl'):
            if not ds_dir.is_dir():
                continue
            prompt_files = sorted(ds_dir.glob('*_prompt.txt'))
            if not prompt_files:
                continue
            print('Found official VerilogEval directory:', ds_dir)
            for prompt_path in prompt_files:
                stem = prompt_path.name[:-len('_prompt.txt')]
                if stem in seen:
                    continue
                seen.add(stem)
                test_path = ds_dir / f'{stem}_test.sv'
                ref_path = ds_dir / f'{stem}_ref.sv'
                prompt = prompt_path.read_text(encoding='utf-8', errors='ignore')
                testbench = test_path.read_text(encoding='utf-8', errors='ignore') if test_path.exists() else ''
                ref_sv = ref_path.read_text(encoding='utf-8', errors='ignore') if ref_path.exists() else ''
                rows.append({
                    'task_id': stem,
                    'prompt': prompt,
                    'module_header': '',
                    'testbench': testbench,
                    'ref_sv': ref_sv,
                    'source_path': str(prompt_path),
                })
    return rows

def _read_json_lines(path):
    rows = []
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise ValueError(f'{path} line {line_no} is not valid JSONL: {exc}') from exc
    return rows

def read_rows(path):
    suffix = path.suffix.lower()
    if suffix == '.jsonl':
        return _read_json_lines(path)
    if suffix == '.json':
        text = path.read_text(encoding='utf-8', errors='ignore').strip()
        try:
            obj = json.loads(text)
        except json.JSONDecodeError:
            # Some RTL benchmark files use .json extension but are actually JSONL.
            return _read_json_lines(path)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            for key in ['data', 'examples', 'tasks', 'benchmark', 'problems']:
                if isinstance(obj.get(key), list):
                    return obj[key]
        raise ValueError(f'Unsupported JSON structure: {path}')
    if suffix == '.csv':
        return pd.read_csv(path).to_dict('records')
    raise ValueError(f'Unsupported benchmark file type: {path}')

def has_prompt_like_fields(rows):
    if not rows:
        return False
    sample = rows[0]
    if not isinstance(sample, dict):
        return False
    keys = set(sample.keys())
    return bool(keys & set(ID_KEYS + PROMPT_KEYS + CONTEXT_KEYS + HEADER_KEYS + TESTBENCH_KEYS))

def row_has_cvdp_category(row):
    cats = row.get('categories', row.get('category', ''))
    if isinstance(cats, str):
        text = cats.lower()
    else:
        text = ' '.join(str(x).lower() for x in cats)
    return any(term in text for term in CVDP_CATEGORY_TERMS)

def load_cvdp_from_hf():
    print(f'Loading CVDP from Hugging Face dataset: {CVDP_HF_DATASET}')
    from datasets import get_dataset_config_names, load_dataset

    configs = CVDP_HF_CONFIGS
    if configs is None:
        configs = get_dataset_config_names(CVDP_HF_DATASET, token=HF_TOKEN)
    print('CVDP configs:', configs)

    rows = []
    load_errors = []
    for config in configs:
        for split in ['eval', 'test', 'validation', 'train']:
            try:
                ds = load_dataset(CVDP_HF_DATASET, config, split=split, token=HF_TOKEN)
                cfg_rows = [dict(row) for row in ds]
                matched = [row for row in cfg_rows if row_has_cvdp_category(row)]
                if matched:
                    print(f'CVDP config={config} split={split}: matched {len(matched)} cid03 rows from {len(cfg_rows)}')
                    rows.extend(matched)
                break
            except Exception as exc:
                load_errors.append(f'{config}/{split}: {type(exc).__name__}: {exc}')
                continue
    if not rows:
        print('CVDP HF load errors / no-match details:')
        for err in load_errors[:30]:
            print(' -', err)
    return rows

def is_missing_value(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.lower() == 'nan':
        return True
    return False

def message_list_to_text(value):
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                role = item.get('role', '')
                content = item.get('content', '')
                if content:
                    parts.append(f'{role}: {content}' if role else str(content))
            else:
                parts.append(str(item))
        return '\n'.join(parts)
    return ''

def first_value(row, keys, default=''):
    for key in keys:
        if key not in row:
            continue
        value = row[key]
        if is_missing_value(value):
            continue
        if isinstance(value, list):
            text = message_list_to_text(value)
            return text if text else json.dumps(value)
        if isinstance(value, dict):
            return json.dumps(value)
        return str(value)
    return default

def normalize_row(row, idx, benchmark):
    task_id = first_value(row, ID_KEYS, f'{benchmark}_{idx}')
    prompt = first_value(row, PROMPT_KEYS, '')
    context = first_value(row, CONTEXT_KEYS, '')
    header = first_value(row, HEADER_KEYS, '')
    testbench = first_value(row, TESTBENCH_KEYS, '')
    if context and context not in prompt:
        prompt = (prompt + '\n\nContext:\n' + context).strip()
    if not prompt:
        excluded = set(TESTBENCH_KEYS + ['patch'])
        prompt = '\n'.join(str(v) for k, v in row.items() if k not in excluded and not is_missing_value(v))
    return {
        'benchmark': benchmark,
        'task_id': task_id,
        'prompt': prompt,
        'module_header': header,
        'testbench': testbench,
        'ref_sv': first_value(row, ['ref_sv'], ''),
        'raw': row,
    }

benchmarks = {}
missing = []
for name, patterns in tqdm(BENCHMARK_PATTERNS.items(), desc='Loading benchmark files'):
    if ACTIVE_BENCHMARK_NAMES and name not in ACTIVE_BENCHMARK_NAMES:
        print(f'Skipping {name}; not needed for this run.')
        continue
    loaded = False
    errors = []

    # Prefer official NVlabs/verilog-eval files for true golden pass@1.
    if name == 'VerilogEval_v2':
        try:
            rows = load_verilogeval_v2_from_official_files()
            if rows:
                norm = [normalize_row(row, i, name) for i, row in enumerate(rows)]
                if MAX_TASKS_PER_BENCHMARK is not None:
                    norm = norm[:MAX_TASKS_PER_BENCHMARK]
                benchmarks[name] = norm
                print(f'Loaded {name}: {len(norm)} rows from official NVlabs/verilog-eval files')
                loaded = True
        except Exception as exc:
            errors.append(f'official NVlabs/verilog-eval files: {type(exc).__name__}: {exc}')

    # Prefer the official Hugging Face dataset for CVDP cid03.
    if (not loaded) and name == 'CVDP_cid03':
        try:
            rows = load_cvdp_from_hf()
            if rows and has_prompt_like_fields(rows):
                norm = [normalize_row(row, i, name) for i, row in enumerate(rows)]
                if MAX_TASKS_PER_BENCHMARK is not None:
                    norm = norm[:MAX_TASKS_PER_BENCHMARK]
                benchmarks[name] = norm
                print(f'Loaded {name}: {len(norm)} rows from Hugging Face {CVDP_HF_DATASET}')
                loaded = True
        except Exception as exc:
            errors.append(f'HuggingFace {CVDP_HF_DATASET}: {type(exc).__name__}: {exc}')

    for path in ([] if loaded else find_candidates(patterns)):
        try:
            rows = read_rows(path)
            if not has_prompt_like_fields(rows):
                errors.append(f'{path}: no recognizable benchmark fields')
                continue
            if name == 'CVDP_cid03':
                rows = [row for row in rows if row_has_cvdp_category(row)]
            if name == 'ChipBench_SC':
                rows = [row for row in rows if 'ChipBench' in str(path)]
                if not rows:
                    errors.append(f'{path}: no cid03/cid003 rows after category filtering')
                    continue
            norm = [normalize_row(row, i, name) for i, row in enumerate(rows)]
            if MAX_TASKS_PER_BENCHMARK is not None:
                norm = norm[:MAX_TASKS_PER_BENCHMARK]
            benchmarks[name] = norm
            print(f'Loaded {name}: {len(norm)} rows from {path}')
            loaded = True
            break
        except Exception as exc:
            errors.append(f'{path}: {type(exc).__name__}: {exc}')
            continue
    if not loaded:
        missing.append(name)
        print(f'MISSING {name}: upload a JSONL/CSV matching {patterns}')
        if errors:
            print('Candidate files rejected:')
            for err in errors[:20]:
                print(' -', err)

if not benchmarks:
    raise FileNotFoundError('No RTL benchmark files found. Enable Internet for auto-clone or upload benchmark JSONL/CSV files.')

if missing:
    print('MISSING benchmarks:', missing)
    print('Searched roots:')
    for root in SEARCH_ROOTS:
        print(' -', root)
    print('Available candidate data files:')
    for root in SEARCH_ROOTS:
        if root.exists():
            for p in sorted(root.rglob('*')):
                if p.is_file() and p.suffix.lower() in {'.jsonl', '.json', '.csv'}:
                    print(' -', p)
    if REQUIRE_ALL_BENCHMARKS:
        raise FileNotFoundError(
            'All four benchmarks are required, but these were missing: ' + ', '.join(missing) +
            '. Upload the missing JSONL/CSV files into the runtime/Drive or add their repo URLs to BENCHMARK_REPOS.'
        )
    print('Continuing with loaded benchmarks only because REQUIRE_ALL_BENCHMARKS=False.')

display(pd.DataFrame([
    {'benchmark': k, 'rows_loaded': len(v), 'has_testbench_rows': sum(bool(x['testbench']) for x in v)}
    for k, v in benchmarks.items()
]))


In [ ]:
# -----------------------------
# Prompting and Verilog extraction
# -----------------------------

SYSTEM_PROMPT = '''You are an expert RTL designer. Generate synthesizable Verilog/SystemVerilog that exactly satisfies the given specification. Output only code.'''

def build_prompt(task):
    header = task.get('module_header', '').strip()
    spec = task.get('prompt', '').strip()
    extra = f'\nRequired module/interface:\n{header}\n' if header else ''
    return f'''{SYSTEM_PROMPT}

Task:
{spec}
{extra}
Rules:
- Output only Verilog/SystemVerilog code.
- Include the complete module implementation.
- Do not include explanations.
'''

def extract_verilog(text):
    text = str(text).strip()
    fence = re.search(r'```(?:systemverilog|verilog|sv)?\s*(.*?)```', text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()
    module_match = re.search(r'\bmodule\b.*?\bendmodule\b', text, flags=re.DOTALL)
    if module_match:
        return module_match.group(0).strip()
    return text

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value))[:160]


In [ ]:
# -----------------------------
# Model loading and generation
# -----------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

def clear_model(model=None):
    try:
        del model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def cleanup_hf_cache():
    # Safe after a model has been fully unloaded. Keeps multi-model runs from
    # accumulating many GB of model shards in Kaggle output/cache space.
    # On Colab we keep cache by default; this avoids downloading the same
    # 7B model repeatedly across the controlled oracle-family ablation.
    if not CLEANUP_HF_CACHE_BETWEEN_MODELS:
        return
    for target in [HF_CACHE_DIR / 'hub', HF_CACHE_DIR / 'transformers']:
        if target.exists():
            shutil.rmtree(target, ignore_errors=True)
            target.mkdir(parents=True, exist_ok=True)

def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=TRUST_REMOTE_CODE,
        token=HF_TOKEN,
    )
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = None
    if LOAD_IN_4BIT:
        if BitsAndBytesConfig is None:
            raise RuntimeError('BitsAndBytesConfig unavailable. Check transformers/bitsandbytes install.')
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map='auto',
        torch_dtype=torch.float16,
        quantization_config=quant_config,
        low_cpu_mem_usage=True,
        trust_remote_code=TRUST_REMOTE_CODE,
        token=HF_TOKEN,
    )
    model.eval()
    return tokenizer, model

def make_chat_or_plain(tokenizer, prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt.replace(SYSTEM_PROMPT, '').strip()},
    ]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

@torch.no_grad()
def generate_one(tokenizer, model, task):
    prompt = build_prompt(task)
    text_prompt = make_chat_or_plain(tokenizer, prompt)
    inputs = tokenizer(text_prompt, return_tensors='pt', truncation=True, max_length=TOKENIZER_MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    do_sample = TEMPERATURE and TEMPERATURE > 0
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=bool(do_sample),
        temperature=float(TEMPERATURE) if do_sample else None,
        top_p=float(TOP_P) if do_sample else None,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0, inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return raw, extract_verilog(raw)


@torch.no_grad()
def generate_text(tokenizer, model, system_prompt, user_prompt, max_new_tokens=None):
    messages = [
        {'role': 'system', 'content': system_prompt.strip()},
        {'role': 'user', 'content': user_prompt.strip()},
    ]
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text_prompt = system_prompt.strip() + '\n\n' + user_prompt.strip()
    inputs = tokenizer(text_prompt, return_tensors='pt', truncation=True, max_length=TOKENIZER_MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    do_sample = TEMPERATURE and TEMPERATURE > 0
    output = model.generate(
        **inputs,
        max_new_tokens=int(max_new_tokens or MAX_NEW_TOKENS),
        do_sample=bool(do_sample),
        temperature=float(TEMPERATURE) if do_sample else None,
        top_p=float(TOP_P) if do_sample else None,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


In [ ]:
# -----------------------------
# Syntax and simulation checks
# -----------------------------

def run_cmd(cmd, timeout=20):
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return proc.returncode, proc.stdout, proc.stderr
    except subprocess.TimeoutExpired as exc:
        return 124, exc.stdout or '', exc.stderr or 'timeout'

def check_verilog(verilog_code, testbench='', work_dir=None, timeout=30, run_simulation=None, reference_code=''):
    if shutil.which('iverilog') is None:
        return {'syntax_available': False, 'syntax_pass': None, 'sim_available': False, 'sim_pass': None, 'stderr': 'iverilog not installed'}
    work_dir = Path(work_dir or OUT_DIR / '_tmp_check')
    work_dir.mkdir(parents=True, exist_ok=True)
    design_path = work_dir / 'design.sv'
    tb_path = work_dir / 'tb.sv'
    ref_path = work_dir / 'ref.sv'
    out_path = work_dir / 'a.vvp'
    design_path.write_text(verilog_code, encoding='utf-8')
    cmd = ['iverilog', '-g2012', '-o', str(out_path), str(design_path)]
    if str(reference_code).strip():
        ref_path.write_text(str(reference_code), encoding='utf-8')
        cmd.append(str(ref_path))
    sim_flag = RUN_SIMULATION if run_simulation is None else bool(run_simulation)
    sim_available = sim_flag and bool(str(testbench).strip())
    if sim_available:
        tb_path.write_text(str(testbench), encoding='utf-8')
        cmd.append(str(tb_path))
    rc, stdout, stderr = run_cmd(cmd, timeout=timeout)
    syntax_pass = rc == 0
    result = {
        'syntax_available': True,
        'syntax_pass': syntax_pass,
        'sim_available': sim_available and syntax_pass and shutil.which('vvp') is not None,
        'sim_pass': None,
        'stdout': stdout,
        'stderr': stderr,
    }
    if result['sim_available']:
        rc2, stdout2, stderr2 = run_cmd(['vvp', str(out_path)], timeout=timeout)
        sim_pass = rc2 == 0
        mismatch = re.search(r'Mismatches:\s*(\d+)\s+in\s+(\d+)\s+samples', stdout2)
        if mismatch:
            sim_pass = sim_pass and int(mismatch.group(1)) == 0
            result['mismatches'] = int(mismatch.group(1))
            result['samples_checked'] = int(mismatch.group(2))
        if 'TIMEOUT' in stdout2 or 'TIMEOUT' in stderr2:
            sim_pass = False
        result['sim_pass'] = sim_pass
        result['stdout'] += '\n' + stdout2
        result['stderr'] += '\n' + stderr2
    return result

def pass_at_k(successes, k):
    # Empirical for generated samples from one task: whether any of first k passed.
    return bool(any(successes[:k]))


In [ ]:
# -----------------------------
# Run baseline model evaluation
# -----------------------------

if RUN_BASELINE_SWEEP:
    # -----------------------------
    # Run evaluation
    # -----------------------------

    all_results = []

    for model_id in MODELS_TO_RUN:
        print('\n' + '=' * 120)
        print('MODEL:', model_id)
        model_slug = safe_name(model_id.replace('/', '__'))
        model_out_dir = OUT_DIR / model_slug
        model_out_dir.mkdir(parents=True, exist_ok=True)

        try:
            tokenizer, model = load_model(model_id)
        except Exception as exc:
            print('MODEL LOAD FAILED:', repr(exc))
            all_results.append({'model': model_id, 'benchmark': '__load__', 'task_id': '', 'sample': 0, 'load_failed': True, 'error': repr(exc)})
            clear_model(None)
            pd.DataFrame(all_results).to_csv(OUT_DIR / 'rtl_eval_results_partial.csv', index=False)
            continue

        try:
            for bench_name, tasks in benchmarks.items():
                print('\nBenchmark:', bench_name, 'tasks:', len(tasks))
                for task in tqdm(tasks, desc=f'{model_id} | {bench_name}', leave=False):
                    for sample_i in range(NUM_SAMPLES_PER_TASK):
                        t0 = time.time()
                        row = {
                            'model': model_id,
                            'benchmark': bench_name,
                            'task_id': task['task_id'],
                            'sample': sample_i,
                            'load_failed': False,
                        }
                        try:
                            raw, verilog = generate_one(tokenizer, model, task)
                            elapsed = time.time() - t0
                            check = check_verilog(
                                verilog,
                                testbench=task.get('testbench', ''),
                                work_dir=model_out_dir / safe_name(f'{bench_name}_{task["task_id"]}_{sample_i}'),
                            )
                            row.update({
                                'elapsed_sec': elapsed,
                                'raw_chars': len(raw),
                                'verilog_chars': len(verilog),
                                **check,
                            })
                            if SAVE_GENERATIONS:
                                gen_path = model_out_dir / f'{safe_name(bench_name)}__{safe_name(task["task_id"])}__s{sample_i}.sv'
                                raw_path = model_out_dir / f'{safe_name(bench_name)}__{safe_name(task["task_id"])}__s{sample_i}.raw.txt'
                                gen_path.write_text(verilog, encoding='utf-8')
                                raw_path.write_text(raw, encoding='utf-8')
                                row['generation_file'] = str(gen_path)
                        except Exception as exc:
                            row.update({'error': repr(exc), 'syntax_pass': False, 'sim_pass': False})
                        all_results.append(row)

                pd.DataFrame(all_results).to_csv(OUT_DIR / 'rtl_eval_results_partial.csv', index=False)
        finally:
            clear_model(model)
            cleanup_hf_cache()

    results = pd.DataFrame(all_results)
    results.to_csv(OUT_DIR / 'rtl_eval_results.csv', index=False)
    display(results.head())
    print('Saved:', OUT_DIR / 'rtl_eval_results.csv')
else:
    print('RUN_BASELINE_SWEEP=False, skipping baseline model sweep.')
    existing = OUT_DIR / 'rtl_eval_results.csv'
    if existing.exists():
        print('Existing baseline results available:', existing)


## VerilogEval Multi-Agent Ablation

This section compares the same RTL generator in two modes:

- **single_agent**: one RTL model generates Verilog, then the benchmark checker runs.
- **multi_agent_initial**: the same RTL is checked by an independently generated oracle/testbench.
- **multi_agent_repaired**: failing cases get one bounded repair round using tool/oracle logs.

The goal is to measure whether the multi-agent verification/repair layer improves VerilogEval pass@1, not merely whether a larger generator is stronger.


In [ ]:
# -----------------------------
# VerilogEval multi-agent ablation
# -----------------------------

def extract_json_object(text):
    text = str(text).strip()
    fenced = re.search(r'```(?:json)?\s*(.*?)```', text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        text = fenced.group(1).strip()
    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        candidate = text[start:end + 1]
        try:
            return json.loads(candidate)
        except Exception:
            pass
    return None


def oracle_system_prompt():
    return '''You are an expert RTL verification engineer.
Return exactly one JSON object and no prose.
The JSON keys must be:
- python_reference: a compact Python behavioral reference as a string.
- systemverilog_testbench: a complete self-checking Verilog/SystemVerilog testbench as a string.
- directed_tests: a list of short descriptions of edge/corner tests.
- metamorphic_checks: a list of invariant/metamorphic properties checked or recommended.
- assertions: a list of assertion/property descriptions.

The testbench must instantiate the DUT module from the required interface when possible.
It should print PASS on success and call $fatal or otherwise exit nonzero on mismatch.
Prefer critical edge cases: reset, zero, max values, signed boundary, overflow, handshake stalls, FSM transitions.
Do not include markdown fences.'''


def build_oracle_prompt(task):
    return f'''RTL specification:
{task.get('prompt', '').strip()}

Required module/interface:
{task.get('module_header', '').strip()}

Generate an independent verification oracle for this task.
The oracle must not rely on any generated RTL implementation.
Output only the requested JSON object.'''


def repair_system_prompt():
    return '''You are an expert RTL designer repairing Verilog/SystemVerilog.
Output only the complete corrected module code. Do not include explanations.'''


def build_repair_prompt(task, rtl, oracle, checks, critic_note=''):
    oracle_tb = (oracle or {}).get('systemverilog_testbench', '')
    return f'''Original RTL specification:
{task.get('prompt', '').strip()}

Required module/interface:
{task.get('module_header', '').strip()}

Previous RTL:
```verilog
{rtl}
```

Generated oracle testbench excerpt:
```verilog
{str(oracle_tb)[:3000]}
```

Verification results/logs:
{json.dumps(checks, indent=2)[:6000]}

Critic guidance:
{critic_note}

Repair the RTL. Preserve the required module interface.'''


def critic_system_prompt():
    return '''You are an independent RTL verification critic.
Given a spec, RTL, oracle/testbench, and logs, decide whether the RTL, oracle/reference, or both are likely wrong.
Return concise JSON with keys verdict and reason. verdict must be one of rtl_bug, oracle_bug, unclear.'''


def build_critic_prompt(task, rtl, oracle, checks):
    return f'''Spec:
{task.get('prompt', '').strip()}

Required module/interface:
{task.get('module_header', '').strip()}

RTL:
```verilog
{rtl[:5000]}
```

Oracle package:
{json.dumps(oracle or {}, indent=2)[:6000]}

Verification logs:
{json.dumps(checks, indent=2)[:6000]}
'''


def candidate_mutants(verilog_code, max_mutants=4):
    # Small mutation set for oracle-quality scoring. Avoid changing nonblocking <=.
    replacements = [
        (r'==', '!=', 'eq_to_neq'),
        (r'!=', '==', 'neq_to_eq'),
        (r'\\+', '-', 'plus_to_minus'),
        (r'&&', '||', 'andand_to_oror'),
        (r'\\|\\|', '&&', 'oror_to_andand'),
        (r'\\b0\\b', '1', 'zero_to_one'),
        (r'\\b1\\b', '0', 'one_to_zero'),
    ]
    mutants = []
    for pattern, repl, name in replacements:
        if len(mutants) >= max_mutants:
            break
        match = re.search(pattern, verilog_code)
        if not match:
            continue
        mutated = verilog_code[:match.start()] + repl + verilog_code[match.end():]
        if mutated != verilog_code:
            mutants.append({'name': name, 'code': mutated})
    return mutants


def task_ref_sv(task):
    return task.get('ref_sv') or task.get('raw', {}).get('ref_sv', '')


def run_oracle_checks(verilog, oracle, task, work_dir):
    oracle = oracle or {}
    tb = str(oracle.get('systemverilog_testbench', '') or '')
    checks = {}
    checks['syntax'] = check_verilog(
        verilog,
        testbench='',
        work_dir=work_dir / 'syntax',
        timeout=30,
        run_simulation=False,
    )
    golden_tb = task.get('testbench', '')
    checks['golden'] = check_verilog(
        verilog,
        testbench=golden_tb,
        work_dir=work_dir / 'golden',
        timeout=60,
        run_simulation=MULTI_AGENT_RUN_GOLDEN_TB and bool(str(golden_tb).strip()),
        reference_code=task_ref_sv(task),
    )
    checks['oracle'] = check_verilog(
        verilog,
        testbench=tb,
        work_dir=work_dir / 'oracle',
        timeout=60,
        run_simulation=MULTI_AGENT_RUN_ORACLE_TB and bool(tb.strip()),
    )
    return checks


def pass_bool(check_dict, key):
    data = check_dict.get(key, {}) if isinstance(check_dict, dict) else {}
    if data.get('sim_available'):
        return bool(data.get('sim_pass'))
    if key == 'syntax':
        return bool(data.get('syntax_pass'))
    return None


def internal_accepts(checks):
    # Internal method signal only. Golden VerilogEval results are evaluation-only
    # and must not decide repair/acceptance in the proposed pipeline.
    syntax_ok = pass_bool(checks, 'syntax')
    oracle_ok = pass_bool(checks, 'oracle')
    if syntax_ok is False:
        return False
    if oracle_ok is False:
        return False
    return True


def repair_reason(checks):
    syntax_ok = pass_bool(checks, 'syntax')
    oracle_ok = pass_bool(checks, 'oracle')
    reasons = []
    if syntax_ok is False:
        reasons.append('compile_failed')
    if oracle_ok is False:
        reasons.append('reference_mismatch')
    return '+'.join(reasons)


def should_repair(checks):
    # If compilation fails, reference simulation cannot be trusted/observed, but
    # the RTL still needs repair. If compilation passes but the generated
    # oracle/reference testbench fails, repair is also justified.
    if MULTI_AGENT_REPAIR_TRIGGER != 'compile_or_reference_mismatch':
        raise ValueError(f'Unsupported repair trigger: {MULTI_AGENT_REPAIR_TRIGGER}')
    return bool(repair_reason(checks))


def mutation_score(verilog, oracle, task, work_dir):
    tb = str((oracle or {}).get('systemverilog_testbench', '') or '')
    if not (MULTI_AGENT_MUTATION_TEST and tb.strip()):
        return {'mutants': 0, 'killed': 0, 'kill_rate': np.nan}
    mutants = candidate_mutants(verilog, MULTI_AGENT_MUTANTS_PER_TASK)
    killed = 0
    for i, mutant in enumerate(tqdm(mutants, desc='Mutation oracle check', leave=False)):
        result = check_verilog(
            mutant['code'],
            testbench=tb,
            work_dir=work_dir / f'mutant_{i}_{safe_name(mutant["name"])}',
            timeout=60,
            run_simulation=True,
        )
        # A useful oracle kills mutants by compile failure or simulation failure.
        killed += int((result.get('syntax_pass') is False) or (result.get('sim_pass') is False))
    total = len(mutants)
    return {'mutants': total, 'killed': killed, 'kill_rate': (killed / total) if total else np.nan}


def compact_check_fields(checks):
    flat = {}
    for prefix, data in checks.items():
        if not isinstance(data, dict):
            continue
        for key in ['syntax_available', 'syntax_pass', 'sim_available', 'sim_pass']:
            flat[f'{prefix}_{key}'] = data.get(key)
        stderr = str(data.get('stderr', '') or '')
        stdout = str(data.get('stdout', '') or '')
        flat[f'{prefix}_stderr_tail'] = stderr[-800:]
        flat[f'{prefix}_stdout_tail'] = stdout[-800:]
    return flat




def select_verilogeval_tasks(tasks, max_tasks=None, mode='first'):
    """Select a larger, more complex VerilogEval subset.

    VerilogEval is roughly ordered from simpler HDLBits-style problems toward
    more complex sequence/FSM-style tasks. This selector avoids measuring only
    the first easy tasks.
    """
    tasks = list(tasks)
    total = len(tasks)
    if max_tasks is None or max_tasks >= total:
        selected = tasks
    elif mode == 'first':
        selected = tasks[:max_tasks]
    elif mode == 'hard_tail':
        selected = tasks[-max_tasks:]
    elif mode == 'complexity_weighted':
        # 20% early, 30% middle, 50% late. This keeps some easy sanity checks
        # but gives the multi-agent repair layer more hard cases to act on.
        n_easy = max(1, int(round(max_tasks * 0.20)))
        n_mid = max(1, int(round(max_tasks * 0.30)))
        n_hard = max_tasks - n_easy - n_mid
        third = max(1, total // 3)
        early = tasks[:third]
        middle = tasks[third:2 * third]
        late = tasks[2 * third:]
        selected = early[:n_easy] + middle[:n_mid] + late[-n_hard:]
    else:
        raise ValueError(f'Unknown MULTI_AGENT_TASK_SELECTION={mode}')
    print(f'Selected {len(selected)} / {total} VerilogEval tasks using mode={mode}')
    if selected:
        print('First selected task:', selected[0]['task_id'])
        print('Last selected task:', selected[-1]['task_id'])
    return selected

if RUN_MULTI_AGENT_VERILOGEVAL:
    if 'VerilogEval_v2' not in benchmarks:
        raise FileNotFoundError('VerilogEval_v2 was not loaded; add VerilogEval/ChipMATE benchmark data first.')

    ma_tasks = select_verilogeval_tasks(
        benchmarks['VerilogEval_v2'],
        max_tasks=MULTI_AGENT_MAX_TASKS,
        mode=MULTI_AGENT_TASK_SELECTION,
    )
    ma_dir = OUT_DIR / 'multi_agent_verilogeval'
    ma_dir.mkdir(parents=True, exist_ok=True)
    ma_rows = []

    oracle_variants = (
        ORACLE_FAMILY_ABLATIONS
        if globals().get('RUN_ORACLE_FAMILY_ABLATION', False)
        else [{
            'name': 'configured_oracle',
            'oracle_model': MULTI_ORACLE_MODEL,
            'family_relation': 'configured',
            'note': 'Single configured oracle model',
        }]
    )

    print('\n' + '=' * 120)
    print('MULTI-AGENT VERILOGEVAL ORACLE-FAMILY ABLATION')
    print('Tasks:', len(ma_tasks))
    print('RTL model:', MULTI_RTL_MODEL)
    print('Repair model:', MULTI_REPAIR_MODEL)
    print('Task selection:', MULTI_AGENT_TASK_SELECTION)
    print('Critic:', MULTI_CRITIC_MODEL if MULTI_AGENT_USE_CRITIC else '(disabled for cleaner oracle-family test)')
    print('Oracle variants:')
    for variant in oracle_variants:
        print(' -', variant['name'], '|', variant.get('family_relation', ''), '|', variant['oracle_model'])

    # Stage A: generate single-agent RTLs once. Every oracle-family condition
    # receives exactly the same generated RTL, so the comparison isolates the
    # verifier/oracle-family effect.
    rtl_generations = {}
    tokenizer, model = load_model(MULTI_RTL_MODEL)
    try:
        for task in tqdm(ma_tasks, desc='Stage A: shared RTL generation'):
            t0 = time.time()
            raw, rtl = generate_one(tokenizer, model, task)
            work = ma_dir / 'single_agent' / safe_name(task['task_id'])
            single_checks = {
                'syntax': check_verilog(rtl, work_dir=work / 'syntax', run_simulation=False),
                'golden': check_verilog(
                    rtl,
                    testbench=task.get('testbench', ''),
                    work_dir=work / 'golden',
                    timeout=60,
                    run_simulation=MULTI_AGENT_RUN_GOLDEN_TB and bool(str(task.get('testbench', '')).strip()),
                    reference_code=task_ref_sv(task),
                ),
            }
            row = {
                'experiment': 'single_agent_baseline',
                'comparison_label': 'single_agent',
                'approach': 'single_agent',
                'model': MULTI_RTL_MODEL,
                'oracle_model': '',
                'oracle_family_relation': 'none',
                'repair_model': '',
                'critic_model': '',
                'benchmark': 'VerilogEval_v2',
                'task_id': task['task_id'],
                'elapsed_sec': time.time() - t0,
                'raw_chars': len(raw),
                'verilog_chars': len(rtl),
                'accepted_internal': pass_bool(single_checks, 'syntax'),
                'pass_at_1_golden': pass_bool(single_checks, 'golden'),
            }
            row.update(compact_check_fields(single_checks))
            ma_rows.append(row)
            rtl_generations[task['task_id']] = {'raw': raw, 'rtl': rtl, 'single_checks': single_checks}
        pd.DataFrame(ma_rows).to_csv(ma_dir / 'multi_agent_ablation_partial.csv', index=False)
    finally:
        clear_model(model)
        cleanup_hf_cache()

    final_rows = []
    variant_delta_rows = []

    for variant in oracle_variants:
        variant_name = safe_name(str(variant['name']))
        oracle_model_id = str(variant['oracle_model'])
        family_relation = str(variant.get('family_relation', 'unknown'))
        variant_dir = ma_dir / variant_name
        variant_dir.mkdir(parents=True, exist_ok=True)

        print('\n' + '-' * 100)
        print('ORACLE FAMILY VARIANT:', variant_name)
        print('Family relation:', family_relation)
        print('Oracle/reference model:', oracle_model_id)
        print('Note:', variant.get('note', ''))

        # Stage B: generate independent oracle/reference packages.
        oracle_packages = {}
        tokenizer, model = load_model(oracle_model_id)
        try:
            for task in tqdm(ma_tasks, desc=f'Stage B: oracle generation | {variant_name}'):
                raw = generate_text(
                    tokenizer,
                    model,
                    oracle_system_prompt(),
                    build_oracle_prompt(task),
                    max_new_tokens=MULTI_AGENT_MAX_NEW_TOKENS_ORACLE,
                )
                oracle = extract_json_object(raw) or {
                    'python_reference': '',
                    'systemverilog_testbench': '',
                    'directed_tests': [],
                    'metamorphic_checks': [],
                    'assertions': [],
                    'parse_error_raw': raw[:3000],
                }
                oracle_packages[task['task_id']] = {'raw': raw, 'oracle': oracle}
            (variant_dir / 'oracle_packages.json').write_text(json.dumps(oracle_packages, indent=2), encoding='utf-8')
        finally:
            clear_model(model)
            cleanup_hf_cache()

        # Stage C: run internal oracle checks against the same initial RTL.
        needs_repair = []
        for task in tqdm(ma_tasks, desc=f'Stage C: oracle verification | {variant_name}'):
            task_id = task['task_id']
            rtl = rtl_generations[task_id]['rtl']
            oracle = oracle_packages[task_id]['oracle']
            work = variant_dir / 'initial' / safe_name(task_id)
            checks = run_oracle_checks(rtl, oracle, task, work)
            mut = mutation_score(rtl, oracle, task, work / 'mutation')
            accepted = internal_accepts(checks)
            golden = pass_bool(checks, 'golden')
            reason = repair_reason(checks)
            row = {
                'experiment': variant_name,
                'comparison_label': f'{variant_name}_initial',
                'approach': 'multi_agent_initial',
                'model': MULTI_RTL_MODEL,
                'oracle_model': oracle_model_id,
                'oracle_family_relation': family_relation,
                'repair_model': '',
                'critic_model': '',
                'benchmark': 'VerilogEval_v2',
                'task_id': task_id,
                'accepted_internal': accepted,
                'pass_at_1_golden': golden,
                'oracle_json_ok': 'parse_error_raw' not in oracle,
                'python_reference_chars': len(str(oracle.get('python_reference', '') or '')),
                'oracle_tb_chars': len(str(oracle.get('systemverilog_testbench', '') or '')),
                'mutation_mutants': mut['mutants'],
                'mutation_killed': mut['killed'],
                'mutation_kill_rate': mut['kill_rate'],
                'repair_triggered': bool(reason),
                'repair_reason': reason,
            }
            row.update(compact_check_fields(checks))
            ma_rows.append(row)
            if should_repair(checks):
                needs_repair.append({'task': task, 'rtl': rtl, 'oracle': oracle, 'checks': checks, 'repair_reason': reason})
        pd.DataFrame(ma_rows).to_csv(ma_dir / 'multi_agent_ablation_partial.csv', index=False)
        print('Needs repair:', len(needs_repair), '/', len(ma_tasks), '| variant:', variant_name)

        # Stage D: optional critic. Off by default for this ablation because the
        # goal is to isolate same-family vs different-family oracle behavior.
        critic_notes = {}
        if MULTI_AGENT_USE_CRITIC and needs_repair:
            print('Loading critic model:', MULTI_CRITIC_MODEL)
            tokenizer, model = load_model(MULTI_CRITIC_MODEL)
            try:
                for item in tqdm(needs_repair, desc=f'Stage D: critic | {variant_name}'):
                    task = item['task']
                    raw = generate_text(
                        tokenizer,
                        model,
                        critic_system_prompt(),
                        build_critic_prompt(task, item['rtl'], item['oracle'], item['checks']),
                        max_new_tokens=MULTI_AGENT_MAX_NEW_TOKENS_CRITIC,
                    )
                    critic_notes[task['task_id']] = raw
            finally:
                clear_model(model)
                cleanup_hf_cache()

        # Stage E: bounded repair. The repair model is identical across oracle
        # variants; only the oracle feedback differs.
        if needs_repair and MULTI_AGENT_MAX_REPAIR_ROUNDS > 0:
            tokenizer, model = load_model(MULTI_REPAIR_MODEL)
            try:
                for item in tqdm(needs_repair, desc=f'Stage E: repair | {variant_name}'):
                    task = item['task']
                    t0 = time.time()
                    raw = generate_text(
                        tokenizer,
                        model,
                        repair_system_prompt(),
                        build_repair_prompt(
                            task,
                            item['rtl'],
                            item['oracle'],
                            item['checks'],
                            critic_note=critic_notes.get(task['task_id'], ''),
                        ),
                        max_new_tokens=MULTI_AGENT_MAX_NEW_TOKENS_RTL,
                    )
                    repaired_rtl = extract_verilog(raw)
                    work = variant_dir / 'repaired' / safe_name(task['task_id'])
                    checks = run_oracle_checks(repaired_rtl, item['oracle'], task, work)
                    mut = mutation_score(repaired_rtl, item['oracle'], task, work / 'mutation')
                    row = {
                        'experiment': variant_name,
                        'comparison_label': f'{variant_name}_repaired',
                        'approach': 'multi_agent_repaired',
                        'model': MULTI_RTL_MODEL,
                        'oracle_model': oracle_model_id,
                        'oracle_family_relation': family_relation,
                        'repair_model': MULTI_REPAIR_MODEL,
                        'critic_model': MULTI_CRITIC_MODEL if MULTI_AGENT_USE_CRITIC else '',
                        'benchmark': 'VerilogEval_v2',
                        'task_id': task['task_id'],
                        'elapsed_sec': time.time() - t0,
                        'raw_chars': len(raw),
                        'verilog_chars': len(repaired_rtl),
                        'accepted_internal': internal_accepts(checks),
                        'pass_at_1_golden': pass_bool(checks, 'golden'),
                        'critic_note': critic_notes.get(task['task_id'], '')[:1200],
                        'repair_input_reason': item.get('repair_reason', ''),
                        'mutation_mutants': mut['mutants'],
                        'mutation_killed': mut['killed'],
                        'mutation_kill_rate': mut['kill_rate'],
                    }
                    row.update(compact_check_fields(checks))
                    ma_rows.append(row)
                pd.DataFrame(ma_rows).to_csv(ma_dir / 'multi_agent_ablation_partial.csv', index=False)
            finally:
                clear_model(model)
                cleanup_hf_cache()

        # Final selected output for this oracle family: use repaired RTL only if
        # it passed internal compile + generated-oracle checks. Golden VerilogEval
        # is not used to choose the final output.
        ma_df_so_far = pd.DataFrame(ma_rows)
        variant_df = ma_df_so_far[ma_df_so_far['experiment'] == variant_name]
        single_df_so_far = ma_df_so_far[ma_df_so_far['approach'] == 'single_agent']

        selected_for_variant = []
        for task in tqdm(ma_tasks, desc=f'Building final output | {variant_name}'):
            task_id = task['task_id']
            task_rows = variant_df[variant_df['task_id'] == task_id]
            initial_rows = task_rows[task_rows['approach'] == 'multi_agent_initial']
            repaired_rows = task_rows[task_rows['approach'] == 'multi_agent_repaired']
            if not len(initial_rows):
                continue
            initial = initial_rows.iloc[-1]
            repaired = repaired_rows.iloc[-1] if len(repaired_rows) else None
            if repaired is not None and bool(repaired.get('accepted_internal')):
                rec = repaired.to_dict()
            else:
                rec = initial.to_dict()
            rec['approach'] = 'multi_agent_final'
            rec['comparison_label'] = f'{variant_name}_final'
            rec['experiment'] = variant_name
            selected_for_variant.append(rec)
            final_rows.append(rec)

        # Per-variant contribution diagnostics.
        final_variant_df = pd.DataFrame(selected_for_variant)
        single_task = single_df_so_far[['task_id', 'accepted_internal', 'pass_at_1_golden']].rename(columns={
            'accepted_internal': 'single_internal_accept',
            'pass_at_1_golden': 'single_golden_pass',
        })
        initial_task = variant_df[variant_df['approach'] == 'multi_agent_initial'][['task_id', 'accepted_internal', 'pass_at_1_golden']].rename(columns={
            'accepted_internal': 'multi_initial_accept',
            'pass_at_1_golden': 'multi_initial_golden_pass',
        })
        repaired_task = variant_df[variant_df['approach'] == 'multi_agent_repaired'][['task_id', 'accepted_internal', 'pass_at_1_golden']].rename(columns={
            'accepted_internal': 'repaired_accept',
            'pass_at_1_golden': 'repaired_golden_pass',
        })
        final_task = final_variant_df[['task_id', 'accepted_internal', 'pass_at_1_golden']].rename(columns={
            'accepted_internal': 'multi_final_accept',
            'pass_at_1_golden': 'multi_final_golden_pass',
        })
        delta = single_task.merge(initial_task, on='task_id', how='left').merge(repaired_task, on='task_id', how='left').merge(final_task, on='task_id', how='left')
        delta['experiment'] = variant_name
        delta['oracle_model'] = oracle_model_id
        delta['oracle_family_relation'] = family_relation
        delta['oracle_caught_single_accept'] = delta['single_internal_accept'].fillna(False).astype(bool) & ~delta['multi_initial_accept'].fillna(False).astype(bool)
        delta['repair_recovered_internal'] = ~delta['multi_initial_accept'].fillna(False).astype(bool) & delta['repaired_accept'].fillna(False).astype(bool)
        if delta['single_golden_pass'].notna().any() and delta['multi_final_golden_pass'].notna().any():
            delta['golden_improved'] = ~delta['single_golden_pass'].fillna(False).astype(bool) & delta['multi_final_golden_pass'].fillna(False).astype(bool)
            delta['golden_regressed'] = delta['single_golden_pass'].fillna(False).astype(bool) & ~delta['multi_final_golden_pass'].fillna(False).astype(bool)
        else:
            delta['golden_improved'] = np.nan
            delta['golden_regressed'] = np.nan
        variant_delta_rows.append(delta)

    ma_df = pd.DataFrame(ma_rows)
    ma_df.to_csv(ma_dir / 'multi_agent_ablation_results.csv', index=False)
    print('Saved:', ma_dir / 'multi_agent_ablation_results.csv')

    internal_diag = ma_df.groupby(['experiment', 'approach', 'oracle_family_relation', 'oracle_model'], dropna=False).agg(
        tasks=('task_id', 'nunique'),
        rows=('task_id', 'size'),
        internal_accept_rate=('accepted_internal', lambda s: float(s.fillna(False).astype(bool).mean())),
        oracle_json_ok_rate=('oracle_json_ok', lambda s: float(s.fillna(False).astype(bool).mean()) if len(s) else np.nan),
        avg_mutation_kill_rate=('mutation_kill_rate', 'mean'),
    ).reset_index()
    internal_diag.to_csv(ma_dir / 'multi_agent_internal_diagnostics.csv', index=False)
    print('Saved:', ma_dir / 'multi_agent_internal_diagnostics.csv')

    # Final pass@1 comparison. This table intentionally contains only the direct
    # single-agent baseline and final selected outputs for each oracle family.
    single_df = ma_df[ma_df['approach'] == 'single_agent']
    final_df = pd.DataFrame(final_rows)
    compare_df = pd.concat([single_df, final_df], ignore_index=True)
    compare_df['comparison_label'] = compare_df['comparison_label'].fillna(compare_df['approach'])
    compare_df['oracle_family_relation'] = compare_df['oracle_family_relation'].fillna('none')
    compare_df['oracle_model'] = compare_df['oracle_model'].fillna('')

    def summarize_final_pass(group):
        # VerilogEval pass@1 denominator is all attempted tasks. A design that
        # fails to compile/simulate counts as a failed sample, not unchecked.
        tasks = int(group['task_id'].nunique())
        if 'golden_sim_available' in group:
            checked_mask = group['golden_sim_available'].fillna(False).astype(bool)
        else:
            checked_mask = group['pass_at_1_golden'].notna()
        checked = int(checked_mask.sum())
        passes_all = group['pass_at_1_golden'].fillna(False).astype(bool)
        pass_count = int(passes_all.sum())
        return pd.Series({
            'tasks': tasks,
            'golden_checked': checked,
            'pass_count': pass_count,
            'pass_at_1': float(pass_count / tasks) if tasks else np.nan,
        })

    compare_summary = compare_df.groupby(
        ['comparison_label', 'oracle_family_relation', 'oracle_model'],
        dropna=False,
    ).apply(summarize_final_pass).reset_index()
    if compare_summary['golden_checked'].sum() == 0 and compare_summary['pass_count'].sum() == 0:
        raise RuntimeError(
            'VerilogEval pass@1 may not have been evaluated: no golden simulations ran. '
            'Rerun benchmark-loading cells and confirm has_testbench_rows > 0 for VerilogEval_v2.'
        )
    compare_summary['pass_at_1_percent'] = (100.0 * compare_summary['pass_at_1']).round(2)
    order = {'single_agent': 0}
    for idx, variant in enumerate(oracle_variants, start=1):
        order[f'{safe_name(str(variant["name"]))}_final'] = idx
    compare_summary['_order'] = compare_summary['comparison_label'].map(order).fillna(99)
    compare_summary = compare_summary.sort_values('_order').drop(columns=['_order']).reset_index(drop=True)

    print('\nVerilogEval pass@1 comparison - oracle-family ablation, final outputs only')
    display(compare_summary)
    compare_summary.to_csv(ma_dir / 'oracle_family_ablation_pass_at_1_summary.csv', index=False)
    compare_summary.to_csv(ma_dir / 'single_vs_multi_final_summary.csv', index=False)
    print('Saved:', ma_dir / 'oracle_family_ablation_pass_at_1_summary.csv')

    if variant_delta_rows:
        delta_all = pd.concat(variant_delta_rows, ignore_index=True)
        delta_all.to_csv(ma_dir / 'oracle_family_ablation_task_deltas.csv', index=False)
        proof_summary = delta_all.groupby(['experiment', 'oracle_family_relation', 'oracle_model'], dropna=False).agg(
            tasks=('task_id', 'nunique'),
            oracle_caught_single_accept=('oracle_caught_single_accept', 'sum'),
            repair_recovered_internal=('repair_recovered_internal', 'sum'),
            golden_improvements=('golden_improved', 'sum'),
            golden_regressions=('golden_regressed', 'sum'),
        ).reset_index()
        proof_summary.to_csv(ma_dir / 'oracle_family_ablation_contribution_summary.csv', index=False)
        print('\nOracle-family contribution summary')
        display(proof_summary)
        print('Saved:', ma_dir / 'oracle_family_ablation_task_deltas.csv')
        print('Saved:', ma_dir / 'oracle_family_ablation_contribution_summary.csv')
else:
    print('RUN_MULTI_AGENT_VERILOGEVAL=False, skipping multi-agent ablation.')




In [ ]:
# -----------------------------
# Summary tables
# -----------------------------

if (OUT_DIR / 'rtl_eval_results.csv').exists():

    results = pd.read_csv(OUT_DIR / 'rtl_eval_results.csv')
    eval_rows = results[~results.get('load_failed', False).fillna(False)].copy()

    def summarize(group):
        n = len(group)
        syntax_avail = group['syntax_available'].fillna(False).astype(bool).sum() if 'syntax_available' in group else 0
        sim_avail = group['sim_available'].fillna(False).astype(bool).sum() if 'sim_available' in group else 0
        return pd.Series({
            'samples': n,
            'syntax_checked': syntax_avail,
            'syntax_pass_rate': group['syntax_pass'].fillna(False).mean() if n else np.nan,
            'sim_checked': sim_avail,
            'sim_pass_rate': group['sim_pass'].fillna(False).mean() if n else np.nan,
            'avg_elapsed_sec': group['elapsed_sec'].mean() if 'elapsed_sec' in group else np.nan,
        })

    summary = eval_rows.groupby(['model', 'benchmark'], dropna=False).apply(summarize).reset_index()
    display(summary)
    summary.to_csv(OUT_DIR / 'rtl_eval_summary.csv', index=False)
    print('Saved:', OUT_DIR / 'rtl_eval_summary.csv')

    # Task-level pass@k when NUM_SAMPLES_PER_TASK > 1.
    if NUM_SAMPLES_PER_TASK > 1:
        pass_rows = []
        for (model, bench, task), group in eval_rows.sort_values('sample').groupby(['model', 'benchmark', 'task_id']):
            syntax_success = group['syntax_pass'].fillna(False).tolist()
            sim_success = group['sim_pass'].fillna(False).tolist() if 'sim_pass' in group else []
            pass_rows.append({
                'model': model,
                'benchmark': bench,
                'task_id': task,
                'syntax_pass_at_k': pass_at_k(syntax_success, NUM_SAMPLES_PER_TASK),
                'sim_pass_at_k': pass_at_k(sim_success, NUM_SAMPLES_PER_TASK) if sim_success else np.nan,
            })
        pass_df = pd.DataFrame(pass_rows)
        display(pass_df.groupby(['model', 'benchmark'])[['syntax_pass_at_k', 'sim_pass_at_k']].mean().reset_index())
else:
    print('No baseline rtl_eval_results.csv found; baseline summary skipped.')
